In [1]:
import pickle
import pandas as pd
import json
import numpy as np
from methods.load_data import LoadData
from methods.preproc import Preproc
from methods.feature_endineering import FeatureEngineering
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings("ignore")

In [2]:
 # или CatBoostRegressor
model = CatBoostClassifier()
model.load_model('models/catboost_model.cbm')

CatBoostClassifier(auto_class_weights='Balanced', depth=7, eval_metric='AUC', iterations=2000, l2_leaf_reg=2, learning_rate=0.01, loss_function='Logloss', random_seed=0, use_best_model=True, verbose=50)

In [3]:
features = model.feature_names_
cat_features = []

In [4]:
with open('models/tickers.pkl', 'rb') as f:
    tickers = pickle.load(f)

In [5]:
# Получаем данные за сегодня после закрытия рынка
# start = (datetime.now()).strftime('%Y-%m-%d')
# print(start)
start = '2026-03-07'
load_cls = LoadData()
df, skipped = load_cls.load_moex_universe(tickers, start_date=start)

# Первичный осмотр данных
df.head()


Loading MOEX universe: 100%|██████████| 69/69 [00:41<00:00,  1.67it/s]


,date,open,high,low,close,volume,ticker
0,2026-03-09,13.250,13.795,13.240,13.703,62814100,AFKS
1,2026-03-10,13.788,13.869,13.450,13.490,58257200,AFKS
2,2026-03-11,13.564,13.715,13.314,13.413,42627100,AFKS
3,2026-03-12,13.400,13.528,13.271,13.333,52491400,AFKS
4,2026-03-13,13.335,13.499,13.320,13.332,38712000,AFKS


In [6]:
df.shape

(1725, 7)

In [7]:
path = f'models/history_dataset.pickle'

with open(path, 'rb') as f:
    df_h = pickle.load(f)

In [8]:
df = pd.concat([df_h, df])

In [9]:
df

,date,open,high,low,close,volume,ticker
2850,2025-10-13,12.400,12.749,12.210,12.373,103647600.0,AFKS
2851,2025-10-14,12.374,12.448,12.130,12.250,40709400.0,AFKS
2852,2025-10-15,12.290,12.440,12.075,12.230,56002600.0,AFKS
2853,2025-10-16,12.230,13.764,12.145,13.538,166151100.0,AFKS
2854,2025-10-17,13.660,13.740,13.167,13.513,115543500.0,AFKS
...,...,...,...,...,...,...,...
1720,2026-04-06,4142.000,4195.500,4102.500,4182.000,1225240.0,YDEX
1721,2026-04-07,4182.000,4243.000,4161.500,4210.000,668661.0,YDEX
1722,2026-04-08,4232.500,4290.000,4217.500,4238.000,665376.0,YDEX
1723,2026-04-09,4249.000,4261.000,4150.000,4162.000,467387.0,YDEX


In [10]:
preproc_cls = Preproc()
df = preproc_cls.preproc(df, threshold=0.3)

In [11]:
df = df.sort_values(by =['ticker', 'date'])
df = df.groupby(['ticker']).tail(100)

path = f'models/history_dataset.pickle'

with open(path, 'wb') as f:
    pickle.dump(df_h, f)

In [12]:
with open("models/features_intervals.json", "r", encoding="utf-8") as f:
    features_intervals = json.load(f)

feature_cls = FeatureEngineering()
df = feature_cls.feature_eng(df, features_intervals, cat_features)

In [13]:
with open("models/threshold.json", "r", encoding="utf-8") as f:
    threshold = json.load(f)

In [23]:
df['probability'] = model.predict_proba(df[features])[:, 1]  # Predict probabilities for the positive class
df['predict'] = (df['probability'] >= threshold).astype(int) #

In [24]:
df = df.sort_values(by=['ticker', 'date'])
result = df.groupby(['ticker']).tail(1)

In [29]:
result = result[['ticker', 'predict', 'probability']]
result['trend'] = np.where(result['predict']==1, 'UP', 'DOWN')
result['probability'] = np.where(result['predict']==1, result['probability'], 1-result['probability'])
result


,ticker,predict,probability,trend
99,AFKS,1,0.519018,UP
199,AFLT,1,0.548701,UP
299,ALRS,0,0.529905,DOWN
399,AQUA,0,0.587122,DOWN
499,ASTR,1,0.608098,UP
...,...,...,...,...
6496,VKCO,1,0.501185,UP
6596,VSEH,1,0.515175,UP
6696,VTBR,1,0.587557,UP
6796,X5,1,0.587068,UP


In [30]:
# Устанавливаем ticker как индекс, выбираем нужные колонки и преобразуем
result_dict = result.set_index('ticker')[['trend', 'probability']].to_dict(orient='index')

In [31]:
result_dict

{'AFKS': {'trend': 'UP', 'probability': 0.5190176378291879},
 'AFLT': {'trend': 'UP', 'probability': 0.5487007417433057},
 'ALRS': {'trend': 'DOWN', 'probability': 0.529904955645865},
 'AQUA': {'trend': 'DOWN', 'probability': 0.587122239880722},
 'ASTR': {'trend': 'UP', 'probability': 0.6080975668365253},
 'BELU': {'trend': 'DOWN', 'probability': 0.5636035125114054},
 'BSPB': {'trend': 'UP', 'probability': 0.5873815453829626},
 'CBOM': {'trend': 'DOWN', 'probability': 0.548141148612487},
 'CHMF': {'trend': 'DOWN', 'probability': 0.5361823817538953},
 'CNRU': {'trend': 'UP', 'probability': 0.5288287707683729},
 'DOMRF': {'trend': 'UP', 'probability': 0.6034933171956279},
 'ELFV': {'trend': 'UP', 'probability': 0.5520553630243628},
 'ENPG': {'trend': 'DOWN', 'probability': 0.591720832619805},
 'EUTR': {'trend': 'DOWN', 'probability': 0.6073814525715322},
 'FEES': {'trend': 'UP', 'probability': 0.5528563771790055},
 'FIXR': {'trend': 'DOWN', 'probability': 0.5258734404350995},
 'FLOT': {'

In [33]:
with open("result/signals.json", "w", encoding="utf-8") as f:
    json.dump(result_dict, f, ensure_ascii=False, indent=4)